# PulseGrid

Self-contained Kaggle notebook for GridPulse tree forecast training.

In [ ]:
import json
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET_FEATURE_COLUMNS = (
    'contracted_md_kw', 'temperature_c', 'humidity_percent', 'hour', 'minute',
    'slot_index', 'day_of_week', 'month', 'season_index', 'is_weekend',
    'is_holiday', 'production_schedule_kw', 'fingerprint_mean_kw',
    'fingerprint_p10_kw', 'fingerprint_p90_kw', 'lag_1', 'lag_4',
    'lag_16', 'lag_96', 'rolling_mean_4', 'rolling_mean_16', 'rolling_mean_96'
)
HORIZON_TO_STEPS = {'1h': 4, '4h': 16, '24h': 96}

def resolve_dataset_root():
    env_root = os.getenv('GRIDPULSE_KAGGLE_DATASET_ROOT')
    if env_root:
        return Path(env_root)
    default = Path('/kaggle/input/gridpulse-forecasting-inputs')
    if default.exists():
        return default
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for candidate in sorted(input_root.iterdir()):
            if candidate.is_dir() and ((candidate / 'run_config.json').exists() or (candidate / 'feeder.csv').exists()):
                return candidate
    return default

def load_run_config(dataset_root):
    candidates = [dataset_root / 'run_config.json']
    if Path('/kaggle/input').exists():
        candidates.extend(sorted(Path('/kaggle/input').glob('*/run_config.json')))
    for candidate in candidates:
        if candidate.exists():
            return json.loads(candidate.read_text())
    return None

def resolve_job(config):
    if not config:
        return {
            'job_name': 'feeder_fd_res_01_1h',
            'dataset_file': 'feeder.csv',
            'entity_type': 'feeder',
            'entity_id': 'FD_RES_01',
            'horizon': '1h',
            'output_name': 'tree_feeder_FD_RES_01_1h',
            'fingerprints_file': 'fingerprints.csv',
        }
    requested_job = os.getenv('GRIDPULSE_JOB_NAME')
    jobs = config.get('jobs', [])
    if requested_job:
        for job in jobs:
            if job.get('job_name') == requested_job:
                return dict(job)
        raise ValueError(f'Job {requested_job} not found in run_config.json')
    return dict(jobs[0])

def season_for_month(month):
    if month in (12, 1, 2):
        return 'winter', 0
    if month in (3, 4):
        return 'spring', 1
    if month in (5, 6):
        return 'summer', 2
    if month in (7, 8, 9):
        return 'monsoon', 3
    return 'autumn', 4

def generate_grid_physics_history(entity_type='feeder', days=45):
    rng = np.random.default_rng(31)
    end = pd.Timestamp('2026-01-31T23:45:00Z')
    timestamps = pd.date_range(end=end, periods=days * 24 * 4, freq='15min', tz='UTC')
    if entity_type == 'substation':
        profile = {'entity_type': 'substation', 'entity_id': 'SS_KTM_01', 'secondary_substation_id': 'SS_KTM_01', 'transformer_id': 'TR_SS_01', 'feeder_id': '', 'feeder_type': 'mixed', 'customer_group_id': '', 'customer_type': 'mixed', 'enterprise_id': '', 'is_dedicated_line': 0, 'contracted_md_kw': 1600.0, 'base_load_kw': 1080.0, 'morning_peak_kw': 120.0, 'evening_peak_kw': 210.0}
    else:
        profile = {'entity_type': 'feeder', 'entity_id': 'FD_RES_01', 'secondary_substation_id': 'SS_KTM_01', 'transformer_id': 'TR_FD_01', 'feeder_id': 'FD_RES_01', 'feeder_type': 'residential', 'customer_group_id': 'CG_RES_01', 'customer_type': 'residential', 'enterprise_id': '', 'is_dedicated_line': 0, 'contracted_md_kw': 520.0, 'base_load_kw': 310.0, 'morning_peak_kw': 34.0, 'evening_peak_kw': 82.0}
    records = []
    for timestamp in timestamps:
        slot_index = int(timestamp.hour * 4 + timestamp.minute // 15)
        is_weekend = int(timestamp.dayofweek >= 5)
        season, season_index = season_for_month(timestamp.month)
        hour_float = timestamp.hour + timestamp.minute / 60.0
        morning = profile['morning_peak_kw'] * np.exp(-0.5 * ((hour_float - 8.0) / 1.6) ** 2)
        evening = profile['evening_peak_kw'] * np.exp(-0.5 * ((hour_float - 19.0) / 2.2) ** 2)
        temperature_c = 18.0 + season_index * 2.0 + 5.5 * np.sin((slot_index / 96.0) * 2 * np.pi) + rng.normal(0, 0.7)
        humidity_percent = float(np.clip(58 + season_index * 5 + rng.normal(0, 4), 35, 95))
        weekend_factor = 0.94 if is_weekend else 1.0
        fingerprint_mean_kw = (profile['base_load_kw'] + morning + evening) * weekend_factor
        weather_adjustment = max(temperature_c - 24.0, 0.0) * 1.2
        load_kw = max(fingerprint_mean_kw + weather_adjustment + rng.normal(0, profile['base_load_kw'] * 0.025), 1.0)
        records.append({**profile, 'timestamp': timestamp.isoformat(), 'load_kw': round(float(load_kw), 3), 'interval_energy_kwh': round(float(load_kw) * 0.25, 3), 'temperature_c': round(float(temperature_c), 3), 'humidity_percent': round(float(humidity_percent), 3), 'day_type': 'weekend' if is_weekend else 'weekday', 'hour': timestamp.hour, 'minute': timestamp.minute, 'slot_index': slot_index, 'month': timestamp.month, 'day_of_week': timestamp.dayofweek, 'season': season, 'season_index': season_index, 'is_weekend': is_weekend, 'is_holiday': 0, 'data_quality_flag': 'synthetic_ok', 'source_type': 'synthetic_ami', 'production_schedule_kw': 0.0, 'fingerprint_mean_kw': round(float(fingerprint_mean_kw), 3), 'fingerprint_p10_kw': round(float(fingerprint_mean_kw) * 0.92, 3), 'fingerprint_p90_kw': round(float(fingerprint_mean_kw) * 1.08, 3)})
    return pd.DataFrame.from_records(records)

def load_inputs(dataset_root, job, config):
    history_path = dataset_root / job.get('dataset_file', 'feeder.csv')
    fingerprint_path = dataset_root / job.get('fingerprints_file', (config or {}).get('default_fingerprints_file', 'fingerprints.csv'))
    if history_path.exists():
        history = pd.read_csv(history_path)
    else:
        print(f'Missing {history_path}; generating fallback grid-physics data')
        history = generate_grid_physics_history(entity_type=job.get('entity_type', 'feeder'))
    if fingerprint_path.exists():
        fingerprints = pd.read_csv(fingerprint_path)
    else:
        fingerprints = history.groupby(['entity_type', 'entity_id', 'day_of_week', 'slot_index'], as_index=False).agg(fingerprint_mean_kw=('load_kw', 'mean'), fingerprint_p10_kw=('load_kw', lambda s: float(s.quantile(0.10))), fingerprint_p90_kw=('load_kw', lambda s: float(s.quantile(0.90))))
    history['timestamp'] = pd.to_datetime(history['timestamp'], utc=True)
    return history, fingerprints

def calculate_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denominator = np.where(np.abs(y_true) < 1e-9, 1e-9, np.abs(y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denominator)) * 100.0)
    r2 = float(r2_score(y_true, y_pred))
    true_peak_index = int(np.argmax(y_true))
    pred_peak_index = int(np.argmax(y_pred))
    return {'mae': round(mae, 6), 'rmse': round(rmse, 6), 'mape': round(mape, 6), 'r2': round(r2, 6), 'peak_time_error': abs(pred_peak_index - true_peak_index), 'peak_load_error': round(float(abs(float(y_pred[pred_peak_index]) - float(y_true[true_peak_index]))), 6)}

def build_feature_frame(history):
    working = history.sort_values('timestamp').reset_index(drop=True).copy()
    shifted_load = working['load_kw'].shift(1)
    working['lag_1'] = working['load_kw'].shift(1)
    working['lag_4'] = working['load_kw'].shift(4)
    working['lag_16'] = working['load_kw'].shift(16)
    working['lag_96'] = working['load_kw'].shift(96)
    working['rolling_mean_4'] = shifted_load.rolling(4, min_periods=1).mean()
    working['rolling_mean_16'] = shifted_load.rolling(16, min_periods=1).mean()
    working['rolling_mean_96'] = shifted_load.rolling(96, min_periods=1).mean()
    return working

def build_training_dataset(feature_frame, horizon_steps):
    rows = []
    base = feature_frame.dropna(subset=['lag_1', 'lag_4', 'lag_16', 'lag_96']).reset_index(drop=True)
    for index in range(len(base) - horizon_steps):
        row = base.iloc[index].to_dict()
        row['target_sequence'] = base['load_kw'].iloc[index + 1:index + 1 + horizon_steps].astype(float).tolist()
        rows.append(row)
    if not rows:
        raise ValueError('No target windows were generated. Increase history length.')
    return pd.DataFrame(rows)

dataset_root = resolve_dataset_root()
config = load_run_config(dataset_root)
job = resolve_job(config)
history, _fingerprints = load_inputs(dataset_root, job, config)
entity_type = str(job.get('entity_type', 'feeder'))
entity_id = str(job.get('entity_id', 'FD_RES_01'))
horizon = str(job.get('horizon', '1h')).lower()
horizon_steps = HORIZON_TO_STEPS[horizon]
target_history = history[(history['entity_type'].astype(str).str.lower() == entity_type.lower()) & (history['entity_id'].astype(str) == entity_id)].copy()
if target_history.empty:
    raise ValueError(f'Unknown target: {entity_type} {entity_id}')
dataset = build_training_dataset(build_feature_frame(target_history), horizon_steps)
split_index = max(1, int(len(dataset) * 0.8))
if split_index >= len(dataset):
    split_index = len(dataset) - 1
train_frame = dataset.iloc[:split_index].copy()
test_frame = dataset.iloc[split_index:].copy()
x_train = train_frame.loc[:, list(TARGET_FEATURE_COLUMNS)]
y_train = np.vstack(train_frame['target_sequence'].to_list())
x_test = test_frame.loc[:, list(TARGET_FEATURE_COLUMNS)]
y_test = np.vstack(test_frame['target_sequence'].to_list())
model = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=7, n_jobs=-1)
model.fit(x_train, y_train)
predictions = model.predict(x_test)
metrics = calculate_metrics(y_test.flatten(), predictions.flatten())
residuals = y_test - predictions
artifact = {'model_name': 'tree', 'entity_type': entity_type, 'entity_id': entity_id, 'horizon': horizon, 'horizon_steps': horizon_steps, 'feature_columns': list(TARGET_FEATURE_COLUMNS), 'model': model, 'metrics': metrics, 'lower_residual': float(np.quantile(residuals, 0.10)), 'upper_residual': float(np.quantile(residuals, 0.90))}
output_dir = Path(os.getenv('GRIDPULSE_KAGGLE_OUTPUT_DIR', '/kaggle/working'))
output_dir.mkdir(parents=True, exist_ok=True)
output_name = job.get('output_name', f'tree_{entity_type}_{entity_id}_{horizon}')
artifact_path = output_dir / f'{output_name}.pkl'
metrics_path = output_dir / f'{output_name}.json'
with artifact_path.open('wb') as artifact_file:
    pickle.dump(artifact, artifact_file)
with metrics_path.open('w', encoding='utf-8') as metrics_file:
    json.dump({'job': job, 'train_rows': int(len(train_frame)), 'test_rows': int(len(test_frame)), 'metrics': metrics}, metrics_file, indent=2)
print(json.dumps({'job': job, 'dataset_root': str(dataset_root), 'artifact_path': str(artifact_path), 'metrics_path': str(metrics_path), 'train_rows': int(len(train_frame)), 'test_rows': int(len(test_frame)), 'metrics': metrics}, indent=2))
